# RAG-Bot: Document Question Answering with RAG

**Project:** Retrieval-Augmented Generation (RAG) over a hostel rulebook PDF  
**Notebook:** Google Colab  

This notebook extracts text from a PDF, splits it into chunks, creates Gemini embeddings, retrieves relevant chunks using cosine similarity, and asks Gemini to answer questions using the retrieved text.

In [9]:
!pip install -q pypdf google-genai

## Step 1: Upload the PDF and extract text

In [10]:
from pypdf import PdfReader

pdf_path = "MRIT_Hostel_Rulebook.pdf"   # uploaded through the Files panel

reader = PdfReader(pdf_path)
text = ""
for page in reader.pages:
    text += (page.extract_text() or "") + "\n"

print("Pages:", len(reader.pages))
print("Total characters extracted:", len(text))

Pages: 5
Total characters extracted: 8664


## Step 2: Chunking (size 800, overlap 150)

In [11]:
def make_chunks(text, chunk_size=800, overlap=150):
    chunks, start = [], 0
    step = chunk_size - overlap
    while start < len(text):
        piece = text[start:start + chunk_size].strip()
        if piece:
            chunks.append(piece)
        start += step
    return chunks

chunks = make_chunks(text, 800, 150)
print("Number of chunks:", len(chunks))
print("\n----- Chunk 5 (full) -----\n")
print(chunks[min(5, len(chunks)-1)])

Number of chunks: 14

----- Chunk 5 (full) -----

tarian mess at the
start of each semester and cannot switch during the semester. Non-vegetarian
items are served on Wednesdays and Sundays only. A mess rebate of Rs. 120

per day is given if a resident is away for 5 or more continuous days, provided
the Mess Secretary is informed in writing at least 2 days before leaving. Taking
food out of the dining hall is not permitted except for students who are ill, who
may take a meal to their room with a doctor's note.
The mess committee meets on the first Monday of every month. It consists of 2
representatives from each block, the Mess Secretary and the Deputy Warden.
Complaints about food quality can be filed in the mess register or by email to
mess@mrit.example.
5. Leave and Outpass Rules
A resident who wishes to leave the campus overnight must


## Step 3: Embeddings (stored together with chunk text)

In [12]:
import numpy as np
import time
from google import genai
from google.genai import types
from google.colab import userdata

client = genai.Client(api_key=userdata.get("GEMINI_API_KEYS"))   # key stored in Colab Secrets

EMBEDDING_MODEL = "gemini-embedding-001"
EMBEDDING_DIMENSION = 768

def embed_texts(texts, task_type):
    vectors = []
    for i in range(0, len(texts), 50):            # send in small batches
        batch = texts[i:i + 50]
        res = client.models.embed_content(
            model=EMBEDDING_MODEL,
            contents=batch,
            config=types.EmbedContentConfig(
                task_type=task_type,
                output_dimensionality=EMBEDDING_DIMENSION,
            ),
        )
        vectors.extend([e.values for e in res.embeddings])
        time.sleep(1)
    arr = np.array(vectors, dtype=float)
    return arr / np.linalg.norm(arr, axis=1, keepdims=True)   # normalise so dot product = cosine similarity

def build_index(chunks):
    return {"chunks": chunks, "embeddings": embed_texts(chunks, "RETRIEVAL_DOCUMENT")}

index = build_index(chunks)
print("Embedding matrix shape:", index["embeddings"].shape)

Embedding matrix shape: (14, 768)


## Step 4: Search step (top 3 by cosine similarity)

In [13]:
def retrieve(question, index, k=3):
    q = embed_texts([question], "RETRIEVAL_QUERY")[0]
    scores = index["embeddings"] @ q
    top = np.argsort(scores)[::-1][:k]
    return [(int(i), float(scores[i]), index["chunks"][i]) for i in top]

## Step 5: Test the search step alone

In [14]:
test_q = "What is the gate closing time for Godavari Block?"   # <-- edit this
for i, score, chunk in retrieve(test_q, index):
    print(f"Chunk #{i} | score = {score:.4f}")
    print(chunk)
    print("-" * 80)

Chunk #3 | score = 0.6889
id for more than 30 days after the due date, the room allotment will
be cancelled.
The security deposit is refunded within 45 days after the resident vacates the
room, after deducting charges for any damage or lost keys. A lost room key
costs Rs. 250 to replace, and a lost ID card costs Rs. 150 to replace.
3. Hostel Timings
 Activity
Timing
Gate closing time (Kaveri and Narmada)
10:00 PM
Gate closing time (Godavari)
9:00 PM
Gate opening time
5:30 AM
Study hours (compulsory for first years)
8:00 PM to 10:00 PM
Lights-out for common areas
11:30 PM
Visitors allowed
Saturday and Sunday, 10:00 AM to 5:00 PM
Residents arriving after gate closing time must sign the late-entry register at the
security desk. Three late entries in a single month will result in a written
warning, and five late entrie
--------------------------------------------------------------------------------
Chunk #4 | score = 0.6722
me must sign the late-entry register at the
security desk. Three l

## Steps 6 & 7: Final answering step

In [15]:
GENERATION_MODEL = "gemini-3.8-flash"      # change to the model used in class if different
NOT_FOUND = "I could not find the answer in the document."

def answer(question, index, k=3):
    results = retrieve(question, index, k)
    context = "\n\n".join(f"[Chunk {i}]\n{c}" for i, s, c in results)
    prompt = (
        "Answer the question using ONLY the text below. "
        f"If the answer is not clearly in the text, reply exactly: \"{NOT_FOUND}\" "
        "Do not use outside knowledge.\n\n"
        f"TEXT:\n{context}\n\nQUESTION: {question}"
    )
    resp = client.models.generate_content(model=GENERATION_MODEL, contents=prompt)
    best_i, best_s, _ = results[0]
    print("Q:", question)
    print("A:", resp.text)
    print(f"Source: chunk #{best_i} | score = {best_s:.4f}   (all: {[(i, round(s,3)) for i,s,_ in results]})")
    print("=" * 80)

## Tests: 3 in-document + 3 out-of-document questions

### My observations (write 2-3 lines in your own words)
Were the answers correct? Were the retrieved chunks the right ones?

- The code executed without errors.
The document was divided into chunks(800:14 chunks;3000:4 chunks).
All 3 in-document questions received answers.
The answers include source chunks and similarity scores.

In [17]:
in_doc = [
    "What is the gate closing time for Godavari Block?",
    "What is the fine for possessing alcohol in the hostel?",
    "How much does a double-sharing room cost per year?",
]
out_doc = [
    "What is the fine for bringing a pet into the hostel?",
    "What are the gym timings in the hostel?",
    "Is there a rule about parking two-wheelers in the hostel?",
]
print("######## IN-DOCUMENT ########\n")
for q in in_doc:
    answer(q, index)
print("\n######## NOT IN DOCUMENT ########\n")
for q in out_doc:
    answer(q, index)

######## IN-DOCUMENT ########

Q: What is the gate closing time for Godavari Block?
A: The gate closing time for Godavari is 9:00 PM (extended to 11:00 PM during examination weeks).
Source: chunk #3 | score = 0.6889   (all: [(3, 0.689), (4, 0.672), (13, 0.633)])
Q: What is the fine for possessing alcohol in the hostel?
A: Based on the text, the penalty is a Rs. 5,000 fine (along with suspension for 1 month).
Source: chunk #10 | score = 0.7823   (all: [(10, 0.782), (11, 0.731), (8, 0.722)])
Q: How much does a double-sharing room cost per year?
A: A double-sharing room costs Rs. 62,000 per year.
Source: chunk #2 | score = 0.7201   (all: [(2, 0.72), (1, 0.694), (13, 0.662)])

######## NOT IN DOCUMENT ########

Q: What is the fine for bringing a pet into the hostel?
A: I could not find the answer in the document.
Source: chunk #10 | score = 0.7133   (all: [(10, 0.713), (11, 0.694), (8, 0.683)])
Q: What are the gym timings in the hostel?
A: I could not find the answer in the document.
Sourc

## Experiment: chunk size 800 -> 3000

### What changed? (write 4-5 lines in your own words)
Cover: number of chunks, scores, whether answers got better/worse, and why.

- When the chunk size changed from 800 to 3000,the number of chunks decreased from 14 to 4.The answers were generated successfully for all questions.The similarity scores were different for each chunk size.Larger chunks contain more information,which may provide better context for answering questions.However,the best chunk size depends on the document and the questions asked.


In [18]:
chunks_big = make_chunks(text, 3000, 150)
index_big = build_index(chunks_big)
print("Chunks at size 800 :", len(chunks))
print("Chunks at size 3000:", len(chunks_big))
print()
for q in in_doc:
    answer(q, index_big)

Chunks at size 800 : 14
Chunks at size 3000: 4

Q: What is the gate closing time for Godavari Block?
A: Based on the text, the gate closing time for Godavari Block is 9:00 PM (extended to 11:00 PM during examination weeks).
Source: chunk #1 | score = 0.6422   (all: [(1, 0.642), (0, 0.635), (3, 0.626)])
Q: What is the fine for possessing alcohol in the hostel?
A: The fine for consuming or possessing alcohol is Rs. 5,000 (along with suspension for 1 month).
Source: chunk #2 | score = 0.7509   (all: [(2, 0.751), (1, 0.699), (0, 0.695)])
Q: How much does a double-sharing room cost per year?
A: A double-sharing room costs Rs. 62,000 per year.
Source: chunk #0 | score = 0.6775   (all: [(0, 0.677), (2, 0.654), (1, 0.642)])
